In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
import pandas as pd
import builtins
import numpy as np


def sliding_windows_id_data(samples, labels, window_size=32, step=2, attack_labels=('T', 'D', 'F', 'S')):
    n = len(samples)

    if isinstance(attack_labels, str):
        attack_labels = (attack_labels,)

    windows = []
    window_labels = []
    next_id_datas = []
    

    for start in range(0, n - window_size, step):
        end = start + window_size
        window_slice = labels[start:end]

        windows.append(samples[start:end])
        next_id_datas.append(samples[end])

        matching_attack = builtins.next(
            (label for label in attack_labels if label in window_slice),
            'R'
        )
        window_labels.append(matching_attack)

    return (
        np.array(windows, dtype=np.uint8),
        np.array(next_id_datas, dtype=object),
        np.array(window_labels, dtype=object)
    )


# ============================================================
# Load and preprocess the ROAD dataset
# ============================================================

def load_road_data(folder_path, window_size=32, step=2):
    """
    Loads the ROAD dataset and returns train/test tensors ready
    for TensorFlow.

    Returns
    -------
    X_train : ndarray
    y_train : ndarray
    X_test : ndarray
    y_test : ndarray
    """

    # -----------------------
    # Load saved arrays
    # -----------------------
    int_id_data = np.load(folder_path + "/int_id_data.npy", allow_pickle=True)
    attack_labels = np.load(folder_path + "/attack_labels.npy", allow_pickle=True)

    int_id_data_test = np.load(folder_path + "/int_id_data_test.npy", allow_pickle=True)
    attack_labels_test = np.load(folder_path + "/attack_labels_test.npy", allow_pickle=True)

    # -----------------------
    # Create sliding windows
    # -----------------------
    X_train = []
    y_train = []

    for i in range(len(int_id_data)):
        windows, _, labels = sliding_windows_id_data(
            int_id_data[i],
            attack_labels[i],
            window_size=window_size,
            step=step
        )

        X_train.extend(windows)
        y_train.extend(labels)

    X_test = []
    y_test = []

    for i in range(len(int_id_data_test)):
        windows, _, labels = sliding_windows_id_data(
            int_id_data_test[i],
            attack_labels_test[i],
            window_size=window_size,
            step=step
        )

        X_test.extend(windows)
        y_test.extend(labels)

    # -----------------------
    # Convert to arrays
    # -----------------------
    X_train = np.array(X_train, dtype=np.float32)
    X_test = np.array(X_test, dtype=np.float32)

    # Add channel dimension
    X_train = X_train[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    # -----------------------
    # One-hot encode labels
    # -----------------------
    y_train = np.array(y_train)
    y_test = np.array(y_test)

    label_map = {"R": 0, "T": 1}

    y_train_onehot = tf.keras.utils.to_categorical(
        [label_map[l] for l in y_train],
        num_classes=2,
    )

    y_test_onehot = tf.keras.utils.to_categorical(
        [label_map[l] for l in y_test],
        num_classes=2,
    )

    return X_train, y_train_onehot, X_test, y_test_onehot


# ============================================================
# Create, train and evaluate the CNN
# ============================================================

def train_cnn(
    X_train,
    y_train,
    X_test,
    y_test,
    epochs=5,
    batch_size=2000,
    conv1_filters=32,
    conv2_filters=64,
    conv1_kernel_size=(3, 3),
    conv2_kernel_size=(3, 3)
):
    """
    Creates, trains and evaluates the CNN.

    Returns
    -------
    model
    history
    """

    model = models.Sequential([
        layers.Input(shape=X_train.shape[1:]),

        layers.Conv2D(conv1_filters, conv1_kernel_size, activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(conv2_filters, conv2_kernel_size, activation="relu"),

        layers.Flatten(),

        layers.Dense(64, activation="relu"),
        layers.Dense(2, activation="softmax"),
    ])

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.2,
    )

        # --------------------------------------------------
    # Predictions
    # --------------------------------------------------
    y_pred_probs = model.predict(X_test, verbose=1)

    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = np.argmax(y_test, axis=1)

    # --------------------------------------------------
    # Metrics
    # --------------------------------------------------
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted")
    recall = recall_score(y_true, y_pred, average="weighted")
    f1 = f1_score(y_true, y_pred, average="weighted")

    report = classification_report(
        y_true,
        y_pred,
        target_names=["R", "T"],
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(y_true, y_pred)

    # False positive / negative rates
    fp = cm.sum(axis=0) - np.diag(cm)
    fn = cm.sum(axis=1) - np.diag(cm)
    tp = np.diag(cm)
    tn = cm.sum() - (fp + fn + tp)

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    metrics = {
        "accuracy": round(accuracy, 4),
        "weighted_precision": round(precision, 4),
        "weighted_recall": round(recall, 4),
        "weighted_f1": round(f1, 4),

        "classification_report": report,
        "confusion_matrix": cm,

        "false_positive_rate": {
            "R": round(float(fpr[0]), 4),
            "T": round(float(fpr[1]), 4),
        },

        "false_negative_rate": {
            "R": round(float(fnr[0]), 4),
            "T": round(float(fnr[1]), 4),
        },
    }

    return model, history, metrics


In [5]:
folder = r"C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\saved_data\ROAD"

X_train, y_train, X_test, y_test = load_road_data(folder)



In [7]:
import pandas as pd

# Hyperparameters to test
conv1_filters_list = [16, 32, 64]
conv2_filters_list = [32, 64, 128]

kernel_sizes = [
    (2, 2),
    (3, 3),
    (5, 5)
]

results = []

for conv1_filters in conv1_filters_list:
    for conv2_filters in conv2_filters_list:
        for conv1_kernel in kernel_sizes:
            for conv2_kernel in kernel_sizes:

                print(
                    f"Testing "
                    f"C1={conv1_filters}, "
                    f"C2={conv2_filters}, "
                    f"K1={conv1_kernel}, "
                    f"K2={conv2_kernel}"
                )

                model, history, metrics = train_cnn(
                    X_train,
                    y_train,
                    X_test,
                    y_test,
                    epochs=2,
                    batch_size=2000,
                    conv1_filters=conv1_filters,
                    conv2_filters=conv2_filters,
                    conv1_kernel_size=conv1_kernel,
                    conv2_kernel_size=conv2_kernel,
                )

                results.append({
                    "conv1_filters": conv1_filters,
                    "conv2_filters": conv2_filters,
                    "conv1_kernel": conv1_kernel,
                    "conv2_kernel": conv2_kernel,
                    "accuracy": metrics["accuracy"],
                    "precision": metrics["weighted_precision"],
                    "recall": metrics["weighted_recall"],
                    "f1": metrics["weighted_f1"],
                })

results_df = pd.DataFrame(results)

# Sort by F1 score (or accuracy if preferred)
results_df = results_df.sort_values("f1", ascending=False)

print(results_df)

Testing C1=16, C2=32, K1=(2, 2), K2=(2, 2)
Epoch 1/2
892/892 ━━━━━━━━━━━━━━━━━━━━ 55s 58ms/step - accuracy: 0.7948 - loss: 0.6735 - val_accuracy: 0.8547 - val_loss: 0.3367
Epoch 2/2
892/892 ━━━━━━━━━━━━━━━━━━━━ 47s 53ms/step - accuracy: 0.9269 - loss: 0.1831 - val_accuracy: 0.8917 - val_loss: 0.2646
17407/17407 ━━━━━━━━━━━━━━━━━━━━ 37s 2ms/step
Testing C1=16, C2=32, K1=(2, 2), K2=(3, 3)
Epoch 1/2
892/892 ━━━━━━━━━━━━━━━━━━━━ 52s 56ms/step - accuracy: 0.8293 - loss: 0.4525 - val_accuracy: 0.8775 - val_loss: 0.2897
Epoch 2/2
892/892 ━━━━━━━━━━━━━━━━━━━━ 53s 59ms/step - accuracy: 0.9532 - loss: 0.1216 - val_accuracy: 0.9691 - val_loss: 0.0814
17407/17407 ━━━━━━━━━━━━━━━━━━━━ 32s 2ms/step
Testing C1=16, C2=32, K1=(2, 2), K2=(5, 5)


ValueError: Computed output size would be zero or negative. Received `inputs shape=(None, 15, 4, 16)`, `kernel shape=(5, 5, 16, 32)`, `dilation_rate=[1 1]`, `strides=(1, 1)`, `padding=valid`.